# Load Dependencies

In [ ]:
import os
import subprocess

import random

# Pickle for pose model
import pickle

# Data processing and collection
import numpy as np
import pandas as pd
import csv

# Image processing
import cv2
import matplotlib.pyplot as plt

# YOLO for object detection
from ultralytics import YOLO

# MediaPipe for pose inference
import mediapipe as mp
from mediapipe.python.solutions.pose import PoseLandmark
from mediapipe.python.solutions.drawing_utils import DrawingSpec # Initializing mediapipe utils

# Tensorflow libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from keras.models import load_model

# Load Models

## Load YOLO Model

In [ ]:
#Load the YOLO Model
model = YOLO("weights/yolov8m.pt")
# model.to('cuda') # If CUDA is available

## Load Pose Classifier (Pickle)

In [ ]:
# # Load trained pose classifier model
# with open('pose_classifier.pkl', 'rb') as f:
#     pose_classifier = pickle.load(f)

## Load Pose Classifier (.h5)

In [ ]:
pose_classifier = tf.keras.models.load_model(r"D:\Projects\Binus\python\PoseDetection\ThesisClean\CollectorTrainerDir\Proposed\pose_classifier.h5")

# MediaPipe Utils

In [ ]:
#Mediapipe pose prediction, tracker and drawer utilities
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_holistic = mp.solutions.holistic
mp_pose = mp.solutions.pose

In [ ]:
# Keypoints variables
# keypoints_head_start = 0 # Pose head start keypoint
# keypoints_head_end = 8 # Pose head end keypoint

keypoints_head = 0

keypoints_body_start = 11 # Pose body start keypoint
keypoints_body_end = 16 # Pose body end keypoint

In [ ]:
# Set amount of confidence value
min_detection_confidence_val = 0.4
min_tracking_confidence_val = 0.4

In [ ]:
custom_style = mp_drawing_styles.get_default_pose_landmarks_style()
custom_connections = list(mp_pose.POSE_CONNECTIONS)

# list of landmarks to exclude from the drawing
excluded_landmarks = [
    mp_pose.PoseLandmark.LEFT_EYE_INNER,
    mp_pose.PoseLandmark.RIGHT_EYE_INNER,
    mp_pose.PoseLandmark.LEFT_EYE,
    mp_pose.PoseLandmark.RIGHT_EYE,
    mp_pose.PoseLandmark.LEFT_EYE_OUTER,
    mp_pose.PoseLandmark.RIGHT_EYE_OUTER,
    mp_pose.PoseLandmark.LEFT_EAR,
    mp_pose.PoseLandmark.RIGHT_EAR,
    mp_pose.PoseLandmark.MOUTH_LEFT,
    mp_pose.PoseLandmark.MOUTH_RIGHT,
    mp_pose.PoseLandmark.LEFT_THUMB,
    mp_pose.PoseLandmark.RIGHT_THUMB,
    mp_pose.PoseLandmark.LEFT_PINKY,
    mp_pose.PoseLandmark.RIGHT_PINKY,
    mp_pose.PoseLandmark.LEFT_INDEX,
    mp_pose.PoseLandmark.RIGHT_INDEX,
    mp_pose.PoseLandmark.LEFT_HIP,
    mp_pose.PoseLandmark.RIGHT_HIP,
    mp_pose.PoseLandmark.LEFT_KNEE,
    mp_pose.PoseLandmark.RIGHT_KNEE,
    mp_pose.PoseLandmark.LEFT_ANKLE,
    mp_pose.PoseLandmark.RIGHT_ANKLE,
    mp_pose.PoseLandmark.LEFT_HEEL,
    mp_pose.PoseLandmark.RIGHT_HEEL,
    mp_pose.PoseLandmark.LEFT_FOOT_INDEX,
    mp_pose.PoseLandmark.RIGHT_FOOT_INDEX
]

for landmark in excluded_landmarks:

    # Change the way the excluded landmarks are drawn
    custom_style[landmark] = DrawingSpec(color=(255,255,0), thickness=None) 
    
    # Remove all connections which contain these landmarks
    custom_connections = [connection_tuple for connection_tuple in custom_connections 
                            if landmark.value not in connection_tuple]

# CSV Creator

In [ ]:
csv_base_filename = "proposed_csv_result"
csv_directory = "csv_result_directory"

In [ ]:
# Create directory if not exist
if not os.path.exists(csv_directory):
    os.makedirs(csv_directory)

In [ ]:
csv_path = f'{csv_directory}/{csv_base_filename}1.csv'

In [ ]:
csv_path

In [ ]:
def create_csv_dir():    
    counter = 1

    # See how many files exist in directory
    while os.path.exists(f'{csv_directory}/{csv_base_filename}{counter}.csv'):
        counter += 1

    # CSV path based on how many file exists
    csv_path = f'{csv_directory}/{csv_base_filename}{counter}.csv'
    print(csv_path)

    # Create the CSV
    with open(csv_path, 'w') as csv_file:
        pass
    
    return csv_path

In [ ]:
diff_name = 'diff_7_testtt'
csv_path = f"D:\Projects\Binus\python\PoseDetection\ThesisClean\csv_result_directory\proposed_csv_{diff_name}.csv"

In [ ]:
# csv_dir_ans = input("Create result collector CSV? (y/n)")

# if csv_dir_ans == "y":  # If y creates a new CSV to save the prediction results
#     csv_path = create_csv_dir()
# else:
#     pass

In [ ]:
header = ["actual_class", "bored", "engaged", "on_the_phone", "raise_hand"]

if not os.path.exists(csv_path):
    with open(csv_path, mode='w', newline='') as f:
        csv_writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
        csv_writer.writerow(header)
        
else:
    print("CSV exists")

In [ ]:
# Result logger

actual_class = "engaged"

# Multi-Person Estimation

In [ ]:
# Open the webcam
# cap = cv2.VideoCapture(f"D:\Projects\Binus\python\PoseDetection\ThesisClean\TestVid\diff\{diff_name}_480.mp4")
cap = cv2.VideoCapture(r"D:\Projects\Binus\python\PoseDetection\ThesisClean\TestVid\1person\1sttrial\raise_hand\raise_hand_1_480.mp4")
# cap = cv2.VideoCapture(0)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Multipose estimation

while True:
    success, image = cap.read()

    if not success:
        print('Error reading frame from the webcam!')
        break

    # Setup YOLO Inference
    yolo_inference = model.predict(image, classes=0, conf = 0.7) # device = 1 for GPU usage

    # Show annotated YOLO frames
    annotated_frame = yolo_inference[0].plot()

    # Bbox coordinates list keeper
    coord_list = []

    # Bounding box (bbox) data extraction
    for result in yolo_inference:
        coord_list.append(result.boxes.xywhn) #XYWH Normalized, only normalized work for bbox extraction

    # Flattened detected bbox values
    flattened_data = [item for sublist in coord_list for item in sublist]
    bbox_df = pd.DataFrame(flattened_data, columns=["center_x", "center_y", "width", "height"]) # Save bbox in dataframe
    
    # Iterate through bbox dataframe
    for idx, annot in bbox_df.iterrows():
        bbox_center_x = annot['center_x'] * width
        bbox_center_y = annot['center_y'] * height
        bbox_width = annot['width'] * width 

        bbox_height = annot['height'] * height

        # Finding top-left and bottom-right bbox coordinates
        bbox_top_left_x = int(bbox_center_x - (bbox_width/2))
        bbox_top_left_y = int(bbox_center_y - (bbox_height/2))
        bbox_bottom_right_x = int(bbox_center_x + (bbox_width/2))
        bbox_bottom_right_y = int(bbox_center_y + (bbox_height/2))

        # Cropping image to bbox
        image_crop = annotated_frame[bbox_top_left_y:bbox_bottom_right_y, bbox_top_left_x:bbox_bottom_right_x]
        
        # Set MediaPipe
        with mp_holistic.Holistic(
                                min_detection_confidence = min_detection_confidence_val,
                                min_tracking_confidence = min_tracking_confidence_val) as holistic:
    
            # Pose estimation
            # Set image as not writeable to improve performance
            image_crop.flags.writeable = False
            image_crop = cv2.cvtColor(image_crop, cv2.COLOR_BGR2RGB)
            results = holistic.process(image_crop)

            # Transpose results to be drawn on the original image
            if results.pose_landmarks is not None:
                for landmark in results.pose_landmarks.landmark:
                    landmark.x = ((abs(bbox_bottom_right_x - bbox_top_left_x) / width) * landmark.x) + (bbox_top_left_x/width)
                    landmark.y = ((abs(bbox_bottom_right_y - bbox_top_left_y) / height) * landmark.y) + (bbox_top_left_y/height)
                                        
                # Draw the pose annotations on the image
                # Set image as writeable
                image.flags.writeable = True
                mp_drawing.draw_landmarks(
                    annotated_frame,
                    results.pose_landmarks,
                    connections = custom_connections, # Remove excluded connections
                    landmark_drawing_spec= custom_style)

            try:
                # Extract Pose landmarks
                pose_head = results.pose_landmarks.landmark[keypoints_head+1] # Head keypoints
                pose_body = results.pose_landmarks.landmark[keypoints_body_start:keypoints_body_end+1] # Body keypoints
                pose = [pose_head] + pose_body # Combined extracted keypoints from pose
                # print(pose)

                row = list(np.array([[landmark.x, landmark.y] for landmark in pose]).flatten())
                # print(row)
                
                X = pd.DataFrame([row])
                # print(X)

                body_language_predict = pose_classifier.predict(X)[0]

                predicted_probabilities = body_language_predict
                predicted_class = np.argmax(body_language_predict)

                if predicted_class == 0:
                    predicted_class = 'bored'
                elif predicted_class == 1:
                    predicted_class = 'engaged'
                elif predicted_class == 2:
                    predicted_class = 'on_the_phone'
                elif predicted_class == 3:
                    predicted_class = 'raise_hand'

                # print(predicted_probabilities)
                # print(predicted_class)

                # For text displayed in the side of the face
                ear_text = 'Status: ' + predicted_class
                
                # Grab ear coords
                coords = tuple(np.multiply(
                                np.array(
                                    (results.pose_landmarks.landmark[mp_holistic.PoseLandmark.LEFT_EAR].x, 
                                    results.pose_landmarks.landmark[mp_holistic.PoseLandmark.LEFT_EAR].y)), 
                                    [640,480]).astype(int))
                
                # Length of background
                cv2.rectangle(annotated_frame, 
                            (coords[0], coords[1]+5), 
                            (coords[0]+len(body_language_predict) * 19, coords[1] - 15),
                            (30, 30, 245), -1)
                
                # Printed ear text (Status)
                cv2.putText(annotated_frame, ear_text, coords,
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)  

                # Append predictions for CSV
                # row.insert(0, predicted_class)
                # row.insert(1, predicted_probabilities)

                predicted_probabilities[0] = round(predicted_probabilities[0], 4)
                predicted_probabilities[1] = round(predicted_probabilities[1], 4)
                predicted_probabilities[2] = round(predicted_probabilities[2], 4)
                predicted_probabilities[3] = round(predicted_probabilities[3], 4)

                row = [actual_class, predicted_probabilities[0], predicted_probabilities[1],
                       predicted_probabilities[2], predicted_probabilities[3]]       
                        
                # print(row)

                # Export predictions to CSV
                with open(csv_path, mode='a', newline='') as f:
                    csv_writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
                    csv_writer.writerow(row)

            except:
                pass

    # Display the frame with pose estimations
    cv2.imshow("YOLOPipe Inference", annotated_frame)
    # cv2.imshow('YOLOPipe Inference', annotated_frame)
    # cv2.resizeWindow("YOLOPipe Inference", 960, 540)


    if cv2.waitKey(30) & 0xFF == ord('q'):  # Press 'q' to exit the loop
        break

# Release the webcam and close all OpenCV windows
cap.release()
cv2.destroyAllWindows()